# Task 3.3 - Evaluate hybrid recommenders with rerankers

This notebook compares two ways of combining the hybrid recommender from Task 1.3 (with the tuned values from Task 1.5) and the Task 3 rerankers:

1. **Individual recommender + reranker:** ItemKNN and SLIMElastic are reranked independently, then evaluated separately.
2. **Individual recommenders, then reranker:** ItemKNN and SLIMElastic scores are fused into the tuned hybrid first; the reranker is applied to that hybrid score matrix afterwards.

All methods use the same Task 1 split, top-100 candidate pool, cutoff `K=10`, and held-out test items. Rerankers never use test labels. The validation split is used only to select the hybrid weight, as in Task 1.5. The final tables report both accuracy and beyond-accuracy metrics so that the accuracy/diversity trade-off is explicit.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from metrics import K, evaluate_beyond_accuracy, evaluate_topk, item_popularity, short_head_mask
from recbole_utils import ROOT, build_model, load_split, model_scores, top_k
from rerankers import mmr, steck_2018_re_rank, xquad, ugf, rerank_all

RESULTS_DIR = ROOT / "group_18" / "results" / "task_3_3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
N_CANDIDATES = 100
LAMBDAS = (0.0, 0.25, 0.5, 0.75, 1.0)
UGF_EPSILONS = (0.05, 0.10)
MODEL_SPECS = {
    "ItemKNN": ("ItemKNN", "ItemKNN", {"k": 100, "shrink": 100}),
    "SLIMElastic": ("SLIMElastic", "SLIMElastic", {"alpha": 0.1, "l1_ratio": 0.1}),
}
RERANKERS = {
    "MMR": (mmr, "genres"),
    "Steck": (steck_2018_re_rank, "genres"),
    "xQuAD": (xquad, "short_head"),
}


## 1. Common split and catalogue context

RecBole uses internal item id `0` for padding. Rerankers therefore receive padded genre and popularity context, while the metrics receive zero-based item ids.

In [2]:
config, dataset, train_data, valid_data, test_data = load_split("SLIMElastic")
UID, IID = dataset.uid_field, dataset.iid_field
N_USERS, N_INTERNAL_ITEMS = dataset.user_num, dataset.item_num
N_ITEMS = N_INTERNAL_ITEMS - 1

train_inter = train_data.dataset.inter_feat
train_users = train_inter[UID].numpy()
train_items = train_inter[IID].numpy() - 1
pop = item_popularity(train_users, train_items, N_ITEMS)
history = pd.Series(train_items).groupby(train_users).apply(np.asarray).to_dict()

item_file = ROOT / "dataset" / "ml-100k" / "ml-100k.item"
item_table = pd.read_csv(item_file, sep="\t")
genre_names = sorted({g for value in item_table["class:token_seq"] for g in str(value).split()})
genre_index = {g: i for i, g in enumerate(genre_names)}
genres_padded = np.zeros((N_INTERNAL_ITEMS, len(genre_names)), dtype=float)
item_token_map = dataset.field2token_id[IID]
for raw_item, genre_value in zip(item_table["item_id:token"], item_table["class:token_seq"]):
    internal_item = item_token_map.get(str(int(raw_item)))
    if internal_item is None:
        raise KeyError(f"Item {raw_item} is missing from RecBole's item-id mapping.")
    for genre in str(genre_value).split():
        genres_padded[internal_item, genre_index[genre]] = 1.0
genres_zero_based = genres_padded[1:]
head_zero_based = short_head_mask(pop, frac=0.2)
head_padded = np.r_[False, head_zero_based]

def held_out_items(eval_data):
    inter = eval_data.dataset.inter_feat
    return pd.Series(inter[IID].numpy() - 1).groupby(inter[UID].numpy()).apply(set).to_dict()

valid_items = held_out_items(valid_data)
test_items = held_out_items(test_data)
eval_users = np.array(sorted(test_items), dtype=int)
valid_users = np.array(sorted(valid_items), dtype=int)
histories = [history.get(int(u), np.array([], dtype=int)) for u in eval_users]
profile_sizes = pd.Series({u: len(h) for u, h in history.items()})
active_threshold = profile_sizes.quantile(0.8)
active_users = np.array([len(history.get(int(u), [])) >= active_threshold for u in eval_users])
user_groups = {"active_users": active_users, "cold_users": ~active_users}
print(f"{N_USERS} users, {N_ITEMS} items, {len(eval_users)} test users, K={K}")

944 users, 1682 items, 943 test users, K=10


## 2. Train the tuned individual recommenders and learn the Task 1.5 hybrid weight

The component settings are the tuned values used in the earlier tasks. The hybrid coefficient is selected on validation by testing `alpha * ItemKNN + (1-alpha) * SLIMElastic`, then fixed before looking at test results.

In [3]:
models, scores_test, scores_valid = {}, {}, {}
for name, (model_name, config_name, overrides) in MODEL_SPECS.items():
    print(f"Training/scoring {name}...")
    net, _, model_dataset, _, model_valid, model_test = build_model(model_name, config_name, **overrides)
    scores_valid[name], valid_model_items = model_scores(net, model_dataset, model_valid)
    scores_test[name], test_model_items = model_scores(net, model_dataset, model_test)
    if set(valid_items) != set(valid_model_items) or set(test_items) != set(test_model_items):
        raise ValueError(f"Held-out users do not align for {name}")
    models[name] = net

def weighted_scores(score_dict, alpha):
    item_scores = np.asarray(score_dict["ItemKNN"], dtype=float)
    slim_scores = np.asarray(score_dict["SLIMElastic"], dtype=float)
    finite = np.isfinite(item_scores) & np.isfinite(slim_scores)
    combined = np.full_like(item_scores, -np.inf, dtype=float)
    combined[finite] = (alpha * item_scores[finite] +
                        (1.0 - alpha) * slim_scores[finite])
    return combined

def topk_recommendations(scores, users, k=K):
    return {int(u): [int(i - 1) for i in top_k(scores, [u], k + 1)[0] if i > 0][:k] for u in users}

weight_rows = []
for alpha in np.linspace(0.0, 1.0, 11):
    hybrid_valid = weighted_scores(scores_valid, alpha)
    result = evaluate_topk(topk_recommendations(hybrid_valid, valid_users), valid_items, ks=(K,))
    weight_rows.append({"alpha": alpha, "mrr@10": result["topk"].loc["mrr", f"@{K}"], "ndcg@10": result["topk"].loc["ndcg", f"@{K}"]})
weight_table = pd.DataFrame(weight_rows)
best_alpha = float(weight_table.loc[weight_table["mrr@10"].idxmax(), "alpha"])
display(weight_table)
print(f"Tuned Task 1.5 alpha = {best_alpha:.2f}; ItemKNN={best_alpha:.2f}, SLIMElastic={1-best_alpha:.2f}")

Training/scoring ItemKNN...


Training/scoring SLIMElastic...


,alpha,mrr@10,ndcg@10
0,0.0,0.416487,0.249512
1,0.1,0.411399,0.242954
2,0.2,0.407105,0.237346
3,0.3,0.406759,0.234774
4,0.4,0.401547,0.230626
5,0.5,0.400711,0.229167
6,0.6,0.395884,0.226762
7,0.7,0.393004,0.224898
8,0.8,0.390549,0.223373
9,0.9,0.388224,0.222450


Tuned Task 1.5 alpha = 0.00; ItemKNN=0.00, SLIMElastic=1.00


## 3. Evaluation helpers

The first pipeline applies a reranker directly to one individual score matrix. The second pipeline creates the tuned weighted hybrid score matrix first and then applies exactly the same reranker and candidate pool. This makes the comparison attributable to the order of fusion and reranking.

In [4]:
def evaluate_lists(recs_internal, label, base_recs=None):
    recs = np.asarray(recs_internal, dtype=int) - 1
    accuracy = evaluate_topk({int(u): r.tolist() for u, r in zip(eval_users, recs)}, test_items, ks=(K,))
    beyond, beyond_pu = evaluate_beyond_accuracy(
        recs, histories, genres_zero_based, pop, N_USERS,
        user_groups=user_groups,
        accuracy=accuracy["per_user"][f"ndcg@{K}"].to_numpy(),
    )
    row = {"pipeline": label}
    row.update({f"accuracy_{k}": v for k, v in accuracy["topk"][f"@{K}"].items()})
    row.update(beyond.to_dict())
    if base_recs is not None:
        base_accuracy = evaluate_topk({int(u): r.tolist() for u, r in zip(eval_users, base_recs - 1)}, test_items, ks=(K,))
        row.update({f"delta_{k}": row[f"accuracy_{k}"] - base_accuracy["topk"].loc[k, f"@{K}"] for k in ("ndcg", "recall", "precision", "mrr")})
    return row, recs, accuracy, beyond, beyond_pu

def reranked_lists(scores, rerank, context, histories_for_rerank=None):
    kwargs = {"genres": genres_padded} if context == "genres" else {"short_head": head_padded}
    return np.asarray(rerank_all(
        rerank, scores, eval_users, K, LAMBDAS[0], n_candidates=N_CANDIDATES,
        histories=histories_for_rerank, **kwargs
    ))

def run_reranker(scores, rerank, context, lam):
    kwargs = {"genres": genres_padded} if context == "genres" else {"short_head": head_padded}
    rerank_histories = {int(u): history.get(int(u), np.array([], dtype=int)) + 1 for u in eval_users} if context == "genres" and rerank is steck_2018_re_rank else None
    return np.asarray(rerank_all(rerank, scores, eval_users, K, lam, n_candidates=N_CANDIDATES, histories=rerank_histories, **kwargs))

def run_ugf(scores, epsilon):
    return np.asarray(ugf(
        scores, eval_users, K, eps=epsilon, in_group_1=active_users,
        n_candidates=N_CANDIDATES,
    ))


## 4. Compare the two construction orders

For readability, the tables retain the reranker parameter. `Base` is the unre-ranked model. Positive accuracy deltas mean an improvement over the corresponding base model; for `mc_kl`, `upd`, `arp`, and `gini`, lower values are preferable.

In [5]:
rows = []
base_lists = {}
for name, scores in {**scores_test, "Weighted_Hybrid": weighted_scores(scores_test, best_alpha)}.items():
    base_lists[name] = np.asarray([[i for i in top_k(scores, [u], K + 1)[0] if i > 0][:K] for u in eval_users])
    row, *_ = evaluate_lists(base_lists[name], f"{name} / Base")
    row["model"] = name
    row["reranker"] = "Base"
    row["lambda"] = 0.0
    rows.append(row)

    for method, (rerank, context) in RERANKERS.items():
        for lam in LAMBDAS:
            recs = run_reranker(scores, rerank, context, lam)
            row, *_ = evaluate_lists(recs, f"{name} / {method}", base_recs=base_lists[name])
            row["model"] = name
            row["reranker"] = method
            row["lambda"] = lam
            rows.append(row)

    for epsilon in UGF_EPSILONS:
        recs = run_ugf(scores, epsilon)
        row, *_ = evaluate_lists(recs, f"{name} / UGF", base_recs=base_lists[name])
        row["model"] = name
        row["reranker"] = "UGF"
        row["lambda"] = epsilon
        rows.append(row)

results = pd.DataFrame(rows)
results.to_csv(RESULTS_DIR / "hybrid_reranker_evaluation.csv", index=False)
accuracy_cols = ["pipeline", "accuracy_ndcg", "accuracy_recall", "accuracy_precision", "accuracy_mrr", "delta_ndcg", "delta_recall", "delta_precision", "delta_mrr"]
display(results[accuracy_cols].round(4))

,pipeline,accuracy_ndcg,accuracy_recall,accuracy_precision,accuracy_mrr,delta_ndcg,delta_recall,delta_precision,delta_mrr
0,ItemKNN / Base,0.2819,0.2402,0.1830,0.4827,NaN,NaN,NaN,NaN
1,ItemKNN / MMR,0.2819,0.2402,0.1830,0.4827,0.0000,0.0000,0.0000,0.0000
2,ItemKNN / MMR,0.2784,0.2343,0.1808,0.4807,-0.0035,-0.0059,-0.0022,-0.0019
3,ItemKNN / MMR,0.2596,0.2176,0.1668,0.4709,-0.0223,-0.0226,-0.0162,-0.0118
4,ItemKNN / MMR,0.2321,0.1848,0.1438,0.4561,-0.0498,-0.0554,-0.0392,-0.0266
5,ItemKNN / MMR,0.2125,0.1559,0.1228,0.4484,-0.0694,-0.0843,-0.0602,-0.0343
6,ItemKNN / Steck,0.2819,0.2402,0.1830,0.4827,0.0000,0.0000,0.0000,0.0000
7,ItemKNN / Steck,0.2782,0.2459,0.1864,0.4601,-0.0038,0.0056,0.0034,-0.0226
8,ItemKNN / Steck,0.2477,0.2388,0.1827,0.3668,-0.0342,-0.0014,-0.0003,-0.1159
9,ItemKNN / Steck,0.2173,0.2252,0.1769,0.2951,-0.0647,-0.0150,-0.0062,-0.1875


## 5. Beyond-accuracy comparison

In [6]:
beyond_cols = ["pipeline", "ild@10", "novelty@10", "mc_kl@10", "upd@10", "aplt@10", "coverage@10", "gini@10", "exposure_ratio@10"]
display(results[beyond_cols].round(4))
hybrid_results = results[results["model"] == "Weighted_Hybrid"].copy()
display(hybrid_results.groupby("reranker")[["accuracy_ndcg", "accuracy_mrr", "ild@10", "mc_kl@10", "aplt@10", "coverage@10"]].mean().round(4))

,pipeline,ild@10,novelty@10,mc_kl@10,upd@10,aplt@10,coverage@10,gini@10,exposure_ratio@10
0,ItemKNN / Base,0.7308,1.9759,0.7764,0.2272,0.0049,0.1540,0.9450,1034.3285
1,ItemKNN / MMR,0.7308,1.9759,0.7764,0.2272,0.0049,0.1540,0.9450,1034.3285
2,ItemKNN / MMR,0.7745,1.9886,0.6356,0.2134,0.0051,0.1576,0.9444,987.1572
3,ItemKNN / MMR,0.8568,2.1343,0.5402,0.1550,0.0261,0.1694,0.9399,199.3467
4,ItemKNN / MMR,0.8998,2.3207,0.5318,0.1204,0.0602,0.1897,0.9344,77.7536
5,ItemKNN / MMR,0.9079,2.4551,0.5089,0.1084,0.0911,0.2140,0.9255,53.1967
6,ItemKNN / Steck,0.7308,1.9759,0.7764,0.2272,0.0049,0.1540,0.9450,1034.3285
7,ItemKNN / Steck,0.7427,1.9800,0.5859,0.2235,0.0055,0.1587,0.9441,818.3996
8,ItemKNN / Steck,0.7478,2.0126,0.4291,0.2036,0.0089,0.1724,0.9406,304.2229
9,ItemKNN / Steck,0.7603,2.1114,0.2561,0.1663,0.0265,0.2021,0.9323,93.7415


,accuracy_ndcg,accuracy_mrr,ild@10,mc_kl@10,aplt@10,coverage@10
reranker,,,,,,
Base,0.3108,0.5113,0.7343,0.6884,0.0066,0.1980
MMR,0.2751,0.4940,0.8373,0.5312,0.0417,0.2132
Steck,0.2484,0.3839,0.7453,0.3822,0.0424,0.2289
UGF,0.3108,0.5113,0.7343,0.6884,0.0066,0.1980
xQuAD,0.3017,0.5002,0.7331,0.6837,0.0652,0.2357


## 6. Discussion of observations

The results should be read in two stages. First, compare each independently reranked ItemKNN/SLIMElastic row with its own `Base` row: this measures the cost and benefit of attaching a reranker to an individual recommender. Second, compare the `Weighted_Hybrid / <reranker>` rows with `Weighted_Hybrid / Base`: this measures the alternative order in which the recommender scores are fused before reranking.

A reranker can improve its target beyond-accuracy metric while lowering accuracy, so the best operating point is not automatically `lambda=1`. MMR is expected to be useful when the hybrid has enough genre variation for the diversity objective to act; Steck is useful when matching the user's genre profile reduces `MC_KL`; and xQuAD is useful when increasing long-tail exposure improves `APLT` or catalogue coverage. If an individual model and the hybrid respond differently, that shows why the order matters: fusion changes the candidate ranking and therefore changes which items are available to the reranker.

The final choice should therefore be the row with the best application-acceptable balance: prefer a Pareto improvement over the relevant base where one exists; otherwise report the largest meaningful beyond-accuracy gain together with its `NDCG@10`, `Recall@10`, and `MRR@10` decrease. The saved CSV contains the exact values used for this discussion.